# 🛡️ SENTINEL — LLM Training on Colab GPU

**Train a Llama-3-8B agent via GRPO on the SENTINEL multi-agent incident-response environment.**

### Before you start:
1. `Runtime → Change runtime type → T4 GPU` (free) or A100 (Colab Pro)
2. Upload your project ZIP **OR** connect GitHub (see Cell 2)
3. Run all cells top to bottom

### What this notebook does:
```
obs (dict)
  → prompt_builder.build_messages()   # obs → chat messages
  → tokenizer.apply_chat_template()   # messages → tokens
  → Llama-3-8B.generate() [4-bit LoRA] # LLM inference on GPU
  → action_parser.parse_llm_action()  # text → Action dict
  → env.step(action) → reward
  → GRPOTrainer.train()               # gradient update
  → ALPCurriculum.record()            # adapt difficulty
```

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — change runtime type!')

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Get the SENTINEL code ─────────────────────────────────────────
# OPTION A — Clone from GitHub (recommended)
GITHUB_REPO = 'SayantikaLaskar/sentinel'  # ← change to your actual repo
GITHUB_BRANCH = 'main'

!git clone --depth 1 --branch {GITHUB_BRANCH} https://github.com/{GITHUB_REPO}.git /content/sentinel
%cd /content/sentinel

# OPTION B — Upload ZIP from your PC
# from google.colab import files
# uploaded = files.upload()  # upload sentinel.zip
# !unzip sentinel.zip -d /content/sentinel
# %cd /content/sentinel

# OPTION C — Google Drive (if you saved the project there)
# from google.colab import drive
# drive.mount('/content/drive')
# %cd '/content/drive/MyDrive/sentinel'

In [ ]:
# ── Cell 2b: Pull latest fixes (action parser + prompt improvements) ──────
!git pull origin main
print('✅ Code updated with latest fixes')
print('   - Lower temperature (0.7 → 0.3) for more deterministic JSON')
print('   - Shorter max_tokens (512 → 256) to prevent rambling')
print('   - Stop strings to halt at closing fence')
print('   - Repair strategy for truncated JSON')
print('   - Prompt pre-fills JSON fence instead of <think>')

In [ ]:
# ── Cell 3: Install dependencies ──────────────────────────────────────────
# Unsloth: optimised Llama fine-tuning (2x faster, 60% less VRAM)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl datasets gymnasium networkx pyyaml hypothesis

# Install SENTINEL itself
!pip install -q -e .

print('\n✅ All dependencies installed')

In [ ]:
# ── Cell 4: Verify SENTINEL environment ───────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

from sentinel.env import Sentinel_Env
from sentinel.training.prompt_builder import build_messages
from sentinel.training.action_parser import parse_llm_action

env = Sentinel_Env(config_path='env_spec.yaml')
obs, _ = env.reset(seed=42)

# Show the prompt the LLM will receive
messages = build_messages(obs, agent_role='holmes', step_number=0)
print('=== SYSTEM PROMPT (first 400 chars) ===')
print(messages[0]['content'][:400])
print('\n=== USER OBSERVATION ===')
print(messages[1]['content'])
print('\n✅ Environment OK')

In [ ]:
# ── Cell 5: Load Llama-3-8B in 4-bit + LoRA ───────────────────────────────
from unsloth import FastLanguageModel
import torch

MODEL_NAME    = 'unsloth/Meta-Llama-3-8B-Instruct'  # 8B fits in T4 (16GB) at 4-bit
# MODEL_NAME  = 'unsloth/Qwen2.5-7B-Instruct'       # Alternative: Qwen2.5-7B
# MODEL_NAME  = 'unsloth/Meta-Llama-3-70B-Instruct-bnb-4bit'  # Needs A100

MAX_SEQ_LEN   = 4096
LORA_R        = 16
LORA_ALPHA    = 32

print(f'Loading {MODEL_NAME} in 4-bit...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = True,
    dtype          = None,   # auto bf16/fp16
)

print('Applying LoRA...')
model = FastLanguageModel.get_peft_model(
    model,
    r                     = LORA_R,
    lora_alpha            = LORA_ALPHA,
    target_modules        = ['q_proj','k_proj','v_proj','o_proj',
                              'gate_proj','up_proj','down_proj'],
    bias                  = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state          = 42,
)

print(f'\n✅ Model loaded | Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M')
print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── Cell 6: Build GRPOTrainer + LLMAgent ──────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

from sentinel.training.llm_agent import LLMAgent, make_grpo_reward_fn
from sentinel.training.pipeline import TrainingConfig
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

AGENT_ROLE  = 'holmes'   # 'holmes' (investigate) or 'forge' (remediate)
EPISODES    = 200        # increase for better results (500+ recommended)
BATCH_SIZE  = 2          # T4: use 2; A100: use 4-8
CKPT_DIR    = '/content/checkpoints'

config = TrainingConfig(
    agent          = AGENT_ROLE,
    model_name     = MODEL_NAME,
    batch_size     = BATCH_SIZE,
    max_steps      = EPISODES,
    lora_r         = LORA_R,
    lora_alpha     = LORA_ALPHA,
    checkpoint_dir = CKPT_DIR,
    log_file       = '/content/training_log.jsonl',
)

# GRPO reward: step reward from SENTINEL's reward function
grpo_reward_fn = make_grpo_reward_fn(env)

# Seed dataset with one sample prompt (GRPOTrainer needs a dataset)
obs0, _ = env.reset(seed=0)
from sentinel.training.prompt_builder import build_messages
seed_msgs = build_messages(obs0, agent_role=AGENT_ROLE, step_number=0)
seed_prompt = tokenizer.apply_chat_template(seed_msgs, tokenize=False, add_generation_prompt=True)
prompt_dataset = Dataset.from_dict({'prompt': [seed_prompt] * BATCH_SIZE})

grpo_cfg = GRPOConfig(
    output_dir                  = CKPT_DIR,
    per_device_train_batch_size = BATCH_SIZE,
    max_steps                   = EPISODES,
    max_new_tokens              = 512,
    temperature                 = 0.7,
    learning_rate               = 5e-6,
    lr_scheduler_type           = 'cosine',
    warmup_steps                = 10,
    logging_steps               = 5,
    save_steps                  = 50,
    bf16                        = True,
    remove_unused_columns       = False,
)

trainer = GRPOTrainer(
    model         = model,
    tokenizer     = tokenizer,
    reward_funcs  = [grpo_reward_fn],
    args          = grpo_cfg,
    train_dataset = prompt_dataset,
)

llm_agent = LLMAgent(
    model      = model,
    tokenizer  = tokenizer,
    agent_role = AGENT_ROLE,
    device     = 'cuda',
)

print(f'✅ GRPOTrainer + LLMAgent ready | agent={AGENT_ROLE} | episodes={EPISODES}')

In [ ]:
# ── Cell 7: Run Training Loop ──────────────────────────────────────────────
import logging
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s', datefmt='%H:%M:%S')

from sentinel.training.pipeline import run_training_loop

print(f'Starting GRPO training: {EPISODES} episodes, agent={AGENT_ROLE}')
print('Each episode: obs → LLM → action → env.step → reward → GRPO update')
print('-' * 60)

all_metrics = run_training_loop(
    trainer       = trainer,
    env           = env,
    config        = config,
    reward_fn     = env.reward_function,
    start_episode = 0,
    llm_agent     = llm_agent,
)

print('\n' + '=' * 60)
print(f'Training complete: {len(all_metrics)} episodes')
last = all_metrics[-10:]
print(f'Last 10 ep avg  : reward={sum(m.total_reward for m in last)/len(last):.3f} | MTTR={sum(m.mttr for m in last)/len(last):.1f}')

In [ ]:
# ── Cell 8: Evaluate trained model ────────────────────────────────────────
from sentinel.training.evaluate import run_evaluation, print_eval_report

print('Running evaluation (20 episodes per difficulty tier)...')
results = run_evaluation(env, env.reward_function, episodes_per_tier=20, seed=42)
print_eval_report(results)

In [ ]:
# ── Cell 9: Save LoRA adapter ─────────────────────────────────────────────
# Saves ONLY the small LoRA delta weights (~80MB), not the full 8B model
SAVE_PATH = f'/content/sentinel_lora_{AGENT_ROLE}'

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

import os
size_mb = sum(
    os.path.getsize(os.path.join(root, f))
    for root, _, files in os.walk(SAVE_PATH)
    for f in files
) / 1e6
print(f'✅ LoRA adapter saved to {SAVE_PATH}  ({size_mb:.0f} MB)')
print('Files:', os.listdir(SAVE_PATH))

In [ ]:
# ── Cell 10: Download results to your PC ──────────────────────────────────
import shutil
from google.colab import files

# Zip the LoRA adapter
shutil.make_archive('/content/sentinel_lora', 'zip', SAVE_PATH)
print('Downloading LoRA adapter zip (~80MB)...')
files.download('/content/sentinel_lora.zip')

# Download training log
if os.path.exists('/content/training_log.jsonl'):
    files.download('/content/training_log.jsonl')
    print('Downloaded training_log.jsonl')

In [ ]:
# ── Cell 11 (optional): Save to Google Drive ──────────────────────────────
# Run this if you want the adapter persisted across Colab sessions

from google.colab import drive
drive.mount('/content/drive')

DRIVE_SAVE = f'/content/drive/MyDrive/sentinel_lora_{AGENT_ROLE}'
shutil.copytree(SAVE_PATH, DRIVE_SAVE, dirs_exist_ok=True)
print(f'✅ LoRA adapter saved to Google Drive: {DRIVE_SAVE}')

In [ ]:
# ── Cell 12 (optional): Load saved adapter on your PC / another session ───
# Run this AFTER downloading sentinel_lora.zip and extracting it

# from unsloth import FastLanguageModel
# from sentinel.training.llm_agent import LLMAgent
#
# LORA_PATH = './sentinel_lora_holmes'   # extracted zip folder
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name     = LORA_PATH,
#     max_seq_length = 4096,
#     load_in_4bit   = True,
# )
# FastLanguageModel.for_inference(model)  # 2x faster inference
#
# agent = LLMAgent(model, tokenizer, agent_role='holmes', device='cuda')
# action = agent.act(obs)   # use in env.step(action)
# print(action)